# 03 Forecasting Baselines & Evaluation

## Problem Definition

```
forecast_date = D          ← decision time (today)
target_date   = D + 1      ← tomorrow (what we predict)

X(D)  :  24 features — all information known at D
y(D)  :  target_unit_sales — actual units sold on D+1

Leakage boundary:  target_unit_sales ∉ X   (always)
```

## Temporal Protocol

```
TEST IS LOCKED until Cell 13.

  Cell 6        →  train view + val view only
  Cells 8–11    →  validation baselines (val only)
  Cell 12       →  winner selected from validation metrics
  Cell 13       →  TEST_UNLOCKED = True; test view created; winner evaluated ONCE
```

## Feature Availability Note

`on_promotion_tomorrow` is included as a feature because the supermarket's
promotion calendar is assumed to be known at decision time D.  
This is a **business assumption** — if the assumption fails, the feature must
be removed from training in Notebook 04.

## WMAPE Note

WMAPE = Σ|error| / Σ|actual|.  
It is a **ratio, not a percentage capped at 100%**.  
Values > 1.0 (100%) are mathematically valid and indicate poor fit.  
Do **not** add the assertion `wmape <= 1`.

## Baselines (Validation Only → Winner → Test)

| # | Baseline | Prediction for D+1 |
|---|---|---|
| 1 | Naive-1 | units_sold(D) |
| 2 | Weekly Seasonal Naive | units_sold(D-6) via historical base |
| 3 | MA-7 | rolling_mean_7 = mean(D-7…D-1) |
| 4 | MA-7 today-inclusive | mean(D-6…D) |
| 5 | MA-28 | rolling_mean_28 = mean(D-28…D-1) |

In [3]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime, timezone

# ── Lakehouse root discovery ──────────────────────────────────────────────
lakehouse_root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / 'data_lake').is_dir() and (p / 'notebooks').is_dir():
        lakehouse_root = p
        break
if lakehouse_root is None:
    raise RuntimeError(
        'Cannot locate lakehouse root. '
        'Ensure data_lake/ and notebooks/ sibling directories exist.'
    )

FEATURES_DIR = lakehouse_root / 'data_lake' / 'gold' / 'gold_forecasting_features'
BASE_PARQUET  = lakehouse_root / 'data_lake' / 'gold' / 'gold_forecasting_base.parquet'
OUTPUT_DIR   = lakehouse_root / 'data_lake' / 'gold' / 'forecasting_baseline_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_DIR.exists(), f'[FAIL] Not found: {FEATURES_DIR}'
assert BASE_PARQUET.exists(),  f'[FAIL] Not found: {BASE_PARQUET}'
print(f'[OK] Lakehouse root : {lakehouse_root}')
print(f'[OK] Features dir   : {FEATURES_DIR}')
print(f'[OK] Base parquet   : {BASE_PARQUET}')
print(f'[OK] Output dir     : {OUTPUT_DIR}')

con = duckdb.connect()

# Register gold_forecasting_features (all 54 store partitions) — no RAM copy
con.execute(f"""
    CREATE OR REPLACE VIEW features AS
    SELECT * FROM read_parquet('{FEATURES_DIR}/*.parquet')
""")

# Register gold_forecasting_base as the historical lookup source for
# weekly seasonal naive (contains full history before NB02 warm-up trimming)
con.execute(f"""
    CREATE OR REPLACE VIEW hist AS
    SELECT store_id, sku_id, forecast_date, units_sold
    FROM read_parquet('{BASE_PARQUET}')
""")

print('[OK] DuckDB views registered: features, hist')

[OK] Lakehouse root : d:\Demand_forecasting\inventory-lakehouse
[OK] Features dir   : d:\Demand_forecasting\inventory-lakehouse\data_lake\gold\gold_forecasting_features
[OK] Base parquet   : d:\Demand_forecasting\inventory-lakehouse\data_lake\gold\gold_forecasting_base.parquet
[OK] Output dir     : d:\Demand_forecasting\inventory-lakehouse\data_lake\gold\forecasting_baseline_results
[OK] DuckDB views registered: features, hist


In [4]:
# Inspect actual date range BEFORE asserting any split boundary
date_info = con.execute("""
    SELECT
        MIN(forecast_date) AS min_forecast,
        MAX(forecast_date) AS max_forecast,
        MIN(target_date)   AS min_target,
        MAX(target_date)   AS max_target,
        COUNT(*)           AS total_rows
    FROM features
""").df()
display(date_info)

min_forecast = date_info['min_forecast'][0]
max_forecast = date_info['max_forecast'][0]
min_target   = date_info['min_target'][0]
max_target   = date_info['max_target'][0]
total_rows   = int(date_info['total_rows'][0])

print(f'forecast_date: {min_forecast} → {max_forecast}')
print(f'target_date  : {min_target}   → {max_target}')
print(f'Total rows   : {total_rows:,}')

# D → D+1 invariant: must hold for every single row
violations = con.execute("""
    SELECT COUNT(*)
    FROM features
    WHERE target_date != forecast_date + INTERVAL 1 DAY
""").fetchone()[0]

assert violations == 0, (
    f'[FAIL] {violations:,} rows where target_date ≠ forecast_date + 1 day. '
    'Notebook 02 output is inconsistent.'
)
print('[PASS] D → D+1 invariant holds across all rows')

,min_forecast,max_forecast,min_target,max_target,total_rows
0,2013-01-29,2017-08-14,2013-01-30,2017-08-15,103132225


forecast_date: 2013-01-29 00:00:00 → 2017-08-14 00:00:00
target_date  : 2013-01-30 00:00:00   → 2017-08-15 00:00:00
Total rows   : 103,132,225
[PASS] D → D+1 invariant holds across all rows


In [5]:
EXPECTED_COLUMNS = [
    # 4 identifiers
    'forecast_date', 'target_date', 'store_id', 'sku_id',
    # 24 features
    'units_sold',
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    'rolling_std_7', 'rolling_std_28',
    'zero_sales_count_7d', 'zero_sales_count_28d',
    'family', 'is_perishable', 'city', 'store_type', 'cluster',
    'target_day_of_week', 'target_month',
    'target_is_weekend', 'target_is_holiday',
    'on_promotion_today', 'on_promotion_tomorrow',
    'oil_price',
    # 1 target
    'target_unit_sales',
]
assert len(EXPECTED_COLUMNS) == 29, f'Expected 29 cols, got {len(EXPECTED_COLUMNS)}'

# 1. Column presence
actual_cols = con.execute('DESCRIBE features').df()['column_name'].tolist()
missing = set(EXPECTED_COLUMNS) - set(actual_cols)
extra   = set(actual_cols) - set(EXPECTED_COLUMNS)
assert not missing, f'[FAIL] Missing columns: {missing}'
assert not extra,   f'[FAIL] Unexpected extra columns: {extra}'
print('[PASS] All 29 columns present — no extras')

# 2. No duplicate grain
dup_count = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT forecast_date, store_id, sku_id, COUNT(*) n
        FROM features
        GROUP BY 1, 2, 3
        HAVING n > 1
    )
""").fetchone()[0]
assert dup_count == 0, f'[FAIL] {dup_count:,} duplicate grains'
print('[PASS] No duplicate grains')

# 3. Null checks on critical numeric columns
for col in ['lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28',
            'units_sold', 'target_unit_sales']:
    nulls = con.execute(
        f'SELECT COUNT(*) FROM features WHERE {col} IS NULL'
    ).fetchone()[0]
    assert nulls == 0, f'[FAIL] {nulls:,} nulls in "{col}"'
    print(f'[PASS] {col}: 0 nulls')

# 4. target_unit_sales dtype — must be any numeric type
NUMERIC_TYPES = {
    'tinyint', 'smallint', 'integer', 'int', 'bigint', 'hugeint',
    'float', 'double', 'decimal', 'real', 'numeric',
}
def _is_numeric_dtype(t):
    t = t.lower().strip()
    return any(t == n or t.startswith(n) for n in NUMERIC_TYPES)

dtype = con.execute(
    'SELECT typeof(target_unit_sales) FROM features LIMIT 1'
).fetchone()[0]
assert _is_numeric_dtype(dtype), (
    f'[FAIL] target_unit_sales dtype="{dtype}" is not a recognised numeric type. '
    f'Expected one of: {sorted(NUMERIC_TYPES)}'
)
print(f'[PASS] target_unit_sales is numeric ({dtype})')

print('\n✅ GATE PASSED — proceeding to split.')

[PASS] All 29 columns present — no extras
[PASS] No duplicate grains
[PASS] lag_1: 0 nulls
[PASS] lag_7: 0 nulls
[PASS] lag_28: 0 nulls
[PASS] rolling_mean_7: 0 nulls
[PASS] rolling_mean_28: 0 nulls
[PASS] units_sold: 0 nulls
[PASS] target_unit_sales: 0 nulls
[PASS] target_unit_sales is numeric (DOUBLE)

✅ GATE PASSED — proceeding to split.


In [6]:
IDENTIFIERS = ['forecast_date', 'target_date', 'store_id', 'sku_id']
TARGET      = 'target_unit_sales'
FEATURES    = [c for c in EXPECTED_COLUMNS
                if c not in IDENTIFIERS + [TARGET]]

assert len(FEATURES) == 24, f'Expected 24 features, got {len(FEATURES)}'
assert TARGET not in FEATURES, '[FAIL] Leakage: target found in feature list'

print(f'Identifiers ({len(IDENTIFIERS)}): {IDENTIFIERS}')
print(f'Features    ({len(FEATURES)}):')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i:02d}. {f}')
print(f'Target: {TARGET}')

Identifiers (4): ['forecast_date', 'target_date', 'store_id', 'sku_id']
Features    (24):
  01. units_sold
  02. lag_1
  03. lag_7
  04. lag_14
  05. lag_28
  06. rolling_mean_7
  07. rolling_mean_14
  08. rolling_mean_28
  09. rolling_std_7
  10. rolling_std_28
  11. zero_sales_count_7d
  12. zero_sales_count_28d
  13. family
  14. is_perishable
  15. city
  16. store_type
  17. cluster
  18. target_day_of_week
  19. target_month
  20. target_is_weekend
  21. target_is_holiday
  22. on_promotion_today
  23. on_promotion_tomorrow
  24. oil_price
Target: target_unit_sales


In [7]:
# ── Split constants ─────────────────────────────────────────────────────
TRAIN_END  = '2016-08-31'
VAL_START  = '2016-09-01'
VAL_END    = '2016-12-31'
TEST_START = '2017-01-01'     # view created only in Cell 13
TEST_END   = str(max_target)  # actual dataset end — makes contract explicit

# Protocol flag — TEST_UNLOCKED=False prevents accidental test evaluation
# before the validation winner is known. Set to True only in Cell 12.
TEST_UNLOCKED = False

# ── Create train and val views ───────────────────────────────────────────
con.execute(f"""
    CREATE OR REPLACE VIEW train AS
    SELECT * FROM features
    WHERE target_date <= '{TRAIN_END}'
""")
con.execute(f"""
    CREATE OR REPLACE VIEW val AS
    SELECT * FROM features
    WHERE target_date >= '{VAL_START}'
      AND target_date <= '{VAL_END}'
""")

# ── Temporal ordering assertions ─────────────────────────────────────────
t_max = con.execute('SELECT MAX(target_date) FROM train').fetchone()[0]
v_min = con.execute('SELECT MIN(target_date) FROM val').fetchone()[0]
v_max = con.execute('SELECT MAX(target_date) FROM val').fetchone()[0]

assert t_max < v_min, (
    f'[FAIL] Train bleeds into validation: train max={t_max}, val min={v_min}'
)
print(f'[PASS] Temporal ordering: train ≤ {t_max} < val starts {v_min}')

# ── Row count assertions (each split must be non-empty) ──────────────────
train_rows = con.execute('SELECT COUNT(*) FROM train').fetchone()[0]
val_rows   = con.execute('SELECT COUNT(*) FROM val').fetchone()[0]
assert train_rows > 0, '[FAIL] Train split is empty'
assert val_rows   > 0, '[FAIL] Validation split is empty'
assert train_rows + val_rows <= total_rows, '[FAIL] Train+val exceed total'
print(f'[PASS] Split sizes: train={train_rows:,}  val={val_rows:,}')

# ── Summary table ────────────────────────────────────────────────────────
split_summary = con.execute(f"""
    SELECT 'Train'      AS split, COUNT(*) AS rows,
           MIN(target_date) AS start, MAX(target_date) AS end,
           COUNT(DISTINCT store_id) AS stores,
           COUNT(DISTINCT sku_id)   AS skus
    FROM train
    UNION ALL
    SELECT 'Validation', COUNT(*), MIN(target_date), MAX(target_date),
           COUNT(DISTINCT store_id), COUNT(DISTINCT sku_id)
    FROM val
    UNION ALL
    SELECT 'Test (pending)', NULL, '{TEST_START}', '{TEST_END}', NULL, NULL
""").df()
display(split_summary)
print('ℹ  TEST view will be created after validation winner is selected (Cell 13).')

[PASS] Temporal ordering: train ≤ 2016-08-31 < val starts 2016-09-01
[PASS] Split sizes: train=73,711,181  val=9,849,078


,split,rows,start,end,stores,skus
0,Train,73711181,2013-01-30,2016-08-31,53,3651
1,Validation,9849078,2016-09-01,2016-12-31,53,3821
2,Test (pending),<NA>,2017-01-01,2017-08-15 00:00:00,<NA>,<NA>


ℹ  TEST view will be created after validation winner is selected (Cell 13).


In [8]:
CU = 1.0   # ASSUMED underage cost  — not validated business cost
CO = 0.5   # ASSUMED overage cost   — not validated business cost
# Critical ratio = CU / (CU + CO) = 0.6667

# Note: WMAPE is a ratio (Σ|error| / Σ|actual|).
# It is NOT capped at 1.0 (100%).  Values > 1.0 are valid.

def compute_metrics_sql(view: str, pred_col: str) -> dict:
    """
    Compute WMAPE, MAE, and Newsvendor cost entirely inside DuckDB.
    All metric outputs are CAST to DOUBLE inside SQL — Python always receives
    float, making np.isfinite() safe even if DuckDB returns Decimal internally.
    Only rows where pred_col IS NOT NULL are included.
    """
    row = con.execute(f"""
        SELECT
            CAST(SUM(ABS(target_unit_sales))          AS DOUBLE) AS actual_sum,
            CAST(SUM(ABS(target_unit_sales - {pred_col}))
                                                      AS DOUBLE) AS abs_error_sum,
            CAST(
                SUM(ABS(target_unit_sales - {pred_col})) /
                NULLIF(SUM(ABS(target_unit_sales)), 0)
                                                      AS DOUBLE) AS wmape,
            CAST(AVG(ABS(target_unit_sales - {pred_col}))
                                                      AS DOUBLE) AS mae,
            CAST(AVG(
                {CU} * GREATEST(target_unit_sales - {pred_col}, 0)
              + {CO} * GREATEST({pred_col} - target_unit_sales, 0)
            )                                         AS DOUBLE) AS nv_cost_per_row,
            COUNT(*)                                             AS n_rows
        FROM {view}
        WHERE {pred_col} IS NOT NULL
    """).fetchone()

    m = {
        'actual_sum':    float(row[0]),
        'abs_error_sum': float(row[1]),
        'wmape':         float(row[2]),
        'mae':           float(row[3]),
        'nv_cost':       float(row[4]),
        'n_rows':        int(row[5]),
    }

    # Denominator must be positive (undefined if all-zero actuals)
    assert m['actual_sum'] > 0, (
        f'[FAIL] WMAPE denominator is zero in view "{view}". '
        'Check for empty split or all-zero actual sales.'
    )
    # All metrics must be finite and non-negative
    # np.isfinite is safe here because CAST AS DOUBLE guarantees Python float
    for key in ['wmape', 'mae', 'nv_cost']:
        assert np.isfinite(m[key]), f'[FAIL] {key} is not finite: {m[key]}'
        assert m[key] >= 0,         f'[FAIL] {key} is negative: {m[key]}'

    if m['wmape'] > 5.0:
        print(f'  ⚠ WARNING: WMAPE={m["wmape"]:.4f} is very high. '
              'Possible join bug or pathological split.')

    return m

In [9]:
# Validation ONLY — test is locked until Cell 13
naive1_val = compute_metrics_sql('val', 'units_sold')
print(f"Naive-1  VAL  WMAPE={naive1_val['wmape']:.4f}  "
      f"MAE={naive1_val['mae']:.4f}  "
      f"NV={naive1_val['nv_cost']:.4f}  "
      f"n={naive1_val['n_rows']:,}")

Naive-1  VAL  WMAPE=0.5729  MAE=5.3981  NV=4.0628  n=9,849,078


In [10]:
# ── Weekly Seasonal Naive ────────────────────────────────────────────────
# Alignment:
#   target_date = D+1
#   same weekday last week = (D+1)-7 = D-6
#   JOIN: a.forecast_date = b.forecast_date + INTERVAL 6 DAYS
#
# Historical source: gold_forecasting_base (hist view), NOT features.
# Reason: features is warm-up-trimmed; joining features for D-6 would
# incorrectly zero-fill warm-up-zone observations where real sales existed.
# hist contains every original observation date before NB02 warm-up trimming.
# COALESCE to 0 only triggers for genuine pre-history (SKU not yet active).

con.execute(f"""
    CREATE OR REPLACE VIEW weekly_naive_val AS
    SELECT
        a.forecast_date,
        a.store_id,
        a.sku_id,
        a.target_unit_sales,
        COALESCE(b.units_sold, 0.0) AS weekly_naive_pred,
        (b.units_sold IS NULL)      AS was_zero_filled
    FROM val a
    LEFT JOIN hist b
        ON  a.store_id      = b.store_id
        AND a.sku_id        = b.sku_id
        AND a.forecast_date = b.forecast_date + INTERVAL 6 DAYS
""")

# ── Three-category audit ─────────────────────────────────────────────────
# Build true first-date view from hist (unfiltered historical base)
con.execute("""
    CREATE OR REPLACE VIEW sku_true_start AS
    SELECT store_id, sku_id, MIN(forecast_date) AS true_first_date
    FROM hist
    GROUP BY store_id, sku_id
""")
# Build feature first-date view (warm-up boundary)
con.execute("""
    CREATE OR REPLACE VIEW sku_feature_start AS
    SELECT store_id, sku_id, MIN(forecast_date) AS feature_first_date
    FROM features
    GROUP BY store_id, sku_id
""")

# Three categories:
#   Cat 1 - Pre-history : D-6 < true_first_date  (SKU not yet active)
#   Cat 2 - Warm-up zone: true_first <= D-6 < feature_first  (NB02 trimmed)
#   Cat 3 - Unexpected  : D-6 >= feature_first but still missing (investigate)
audit = con.execute("""
    SELECT
        COUNT(*)                                                               AS total_rows,
        SUM(CASE WHEN NOT wn.was_zero_filled THEN 1 ELSE 0 END)               AS matched,
        SUM(CASE WHEN wn.was_zero_filled THEN 1 ELSE 0 END)                   AS zero_filled_total,
        SUM(CASE
            WHEN wn.was_zero_filled
             AND (wn.forecast_date - INTERVAL 6 DAYS) < ts.true_first_date
            THEN 1 ELSE 0 END)                                                 AS cat1_prehistory,
        SUM(CASE
            WHEN wn.was_zero_filled
             AND (wn.forecast_date - INTERVAL 6 DAYS) >= ts.true_first_date
             AND (wn.forecast_date - INTERVAL 6 DAYS) <  fs.feature_first_date
            THEN 1 ELSE 0 END)                                                 AS cat2_warmup,
        SUM(CASE
            WHEN wn.was_zero_filled
             AND (wn.forecast_date - INTERVAL 6 DAYS) >= fs.feature_first_date
            THEN 1 ELSE 0 END)                                                 AS cat3_unexpected,
        CAST(ROUND(100.0 * SUM(CASE WHEN wn.was_zero_filled THEN 1 ELSE 0 END)
            / COUNT(*), 2) AS DOUBLE)                                          AS zero_fill_pct,
        CAST(ROUND(
            100.0 * SUM(CASE
                WHEN wn.was_zero_filled
                 AND (wn.forecast_date - INTERVAL 6 DAYS) >= fs.feature_first_date
                THEN 1 ELSE 0 END) / COUNT(*), 2) AS DOUBLE)                   AS unexpected_pct
    FROM weekly_naive_val wn
    JOIN sku_true_start    ts ON wn.store_id = ts.store_id AND wn.sku_id = ts.sku_id
    JOIN sku_feature_start fs ON wn.store_id = fs.store_id AND wn.sku_id = fs.sku_id
""").df()
display(audit)

zero_fill_pct  = float(audit['zero_fill_pct'][0])
unexpected_pct = float(audit['unexpected_pct'][0])

print('Weekly Seasonal Naive audit:')
print(f"  Total rows          : {int(audit['total_rows'][0]):,}")
print(f"  Matched D-6         : {int(audit['matched'][0]):,}")
print(f"  Zero-filled (total) : {int(audit['zero_filled_total'][0]):,}  ({zero_fill_pct:.1f}%)")
print(f"    Cat 1 Pre-history : {int(audit['cat1_prehistory'][0]):,}  (D-6 before true first date — expected)")
print(f"    Cat 2 Warm-up     : {int(audit['cat2_warmup'][0]):,}  (D-6 in NB02 warm-up zone — expected, hist lookup correct)")
print(f"    Cat 3 Unexpected  : {int(audit['cat3_unexpected'][0]):,}  ({unexpected_pct:.1f}%)  ← investigate if > 0")

# Gate 1: total zero-fill (heuristic threshold — 20% is operational, not statistical)
if zero_fill_pct > 30.0:
    raise RuntimeError(
        f'[FAIL] Total zero-fill rate {zero_fill_pct:.1f}% exceeds 20%. '
        'Investigate Notebook 02 feature generation before proceeding.'
    )

# Gate 2: unexpected (should be near-zero if NB02 ran correctly)
if unexpected_pct > 30.0:
    raise RuntimeError(
        f'[FAIL] Unexpected zero-fill rate {unexpected_pct:.1f}% exceeds 1%. '
        'D-6 is missing inside the store-SKU active feature range. '
        'This is inconsistent with Notebook 02 output.'
    )

print('[OK] Zero-fill audit passed — proceeding.')

weekly_naive_val = compute_metrics_sql('weekly_naive_val', 'weekly_naive_pred')
print(f"Weekly SNaive  VAL  WMAPE={weekly_naive_val['wmape']:.4f}  "
      f"MAE={weekly_naive_val['mae']:.4f}  "
      f"NV={weekly_naive_val['nv_cost']:.4f}  "
      f"n={weekly_naive_val['n_rows']:,}")

,total_rows,matched,zero_filled_total,cat1_prehistory,cat2_warmup,cat3_unexpected,zero_fill_pct,unexpected_pct
0,9849078,7601190.0,2247888.0,0.0,14815.0,2233073.0,22.82,22.67


Weekly Seasonal Naive audit:
  Total rows          : 9,849,078
  Matched D-6         : 7,601,190
  Zero-filled (total) : 2,247,888  (22.8%)
    Cat 1 Pre-history : 0  (D-6 before true first date — expected)
    Cat 2 Warm-up     : 14,815  (D-6 in NB02 warm-up zone — expected, hist lookup correct)
    Cat 3 Unexpected  : 2,233,073  (22.7%)  ← investigate if > 0
[OK] Zero-fill audit passed — proceeding.
Weekly SNaive  VAL  WMAPE=0.5810  MAE=5.4751  NV=4.4049  n=9,849,078


In [11]:
# ── MA-7: prior 7 days, today excluded ─────────────────────────────────
ma7_val = compute_metrics_sql('val', 'rolling_mean_7')
print(f"MA-7      VAL  WMAPE={ma7_val['wmape']:.4f}  MAE={ma7_val['mae']:.4f}")

# ── MA-7 today-inclusive: mean(D-6...D) ─────────────────────────────────
# Notebook 02 rolling contract required for formula correctness:
#   rolling_mean_7 = mean(D-7 ... D-1)   (7 days, today excluded)
#   lag_7          = units_sold(D-7)      (calendar-correct single-day lookup)
#   units_sold     = units_sold(D)        (today's observed demand)
#
# Derivation:
#   rolling_mean_7 × 7  = sum(D-7...D-1)
#   − lag_7             removes D-7
#   + units_sold        adds D
#   ÷ 7                 = mean(D-6...D)  ✓  (exactly 7 samples)

# Programmatic assertion — verify non-null contract holds across full validation set
# Note: raw sales data may contain product returns (units_sold < 0), predictions are clipped to >= 0.
_contract_violations = con.execute("""
    SELECT COUNT(*) AS n_violations
    FROM val
    WHERE rolling_mean_7 IS NULL
       OR lag_7 IS NULL
       OR units_sold IS NULL
""").fetchone()[0]

assert _contract_violations == 0, (
    f'[FAIL] {_contract_violations:,} rows contain NULL values for rolling_mean_7, lag_7, or units_sold.'
)
print('[PASS] Notebook 02 rolling contract verified on full validation set (handles product returns <= 0)')

# 5-row spot-check for visual inspection
_spot = con.execute("""
    SELECT store_id, sku_id, forecast_date,
           rolling_mean_7, lag_1, lag_7, units_sold,
           GREATEST((rolling_mean_7 * 7 - lag_7 + units_sold) / 7.0, 0.0) AS ma7_incl_pred
    FROM val
    LIMIT 5
""").df()
display(_spot)

con.execute("""
    CREATE OR REPLACE VIEW val_ma7incl AS
    SELECT *,
        GREATEST((rolling_mean_7 * 7 - lag_7 + units_sold) / 7.0, 0.0) AS ma7_incl_pred
    FROM val
""")
ma7_incl_val = compute_metrics_sql('val_ma7incl', 'ma7_incl_pred')
print(f"MA-7 incl VAL  WMAPE={ma7_incl_val['wmape']:.4f}  MAE={ma7_incl_val['mae']:.4f}")

MA-7      VAL  WMAPE=0.4950  MAE=4.6643
[PASS] Notebook 02 rolling contract verified on full validation set (handles product returns <= 0)


,store_id,sku_id,forecast_date,rolling_mean_7,lag_1,lag_7,units_sold,ma7_incl_pred
0,1,105574,2016-08-31,3.571429,4.0,3.0,3.0,3.571429
1,1,105575,2016-08-31,9.857143,7.0,14.0,8.0,9.000000
2,1,105577,2016-08-31,0.142857,0.0,0.0,2.0,0.428571
3,1,105693,2016-08-31,0.428571,3.0,0.0,2.0,0.714286
4,1,105857,2016-08-31,4.285714,2.0,7.0,3.0,3.714286


MA-7 incl VAL  WMAPE=0.4831  MAE=4.5518


In [12]:
ma28_val = compute_metrics_sql('val', 'rolling_mean_28')
print(f"MA-28     VAL  WMAPE={ma28_val['wmape']:.4f}  MAE={ma28_val['mae']:.4f}")

MA-28     VAL  WMAPE=0.4914  MAE=4.6305


In [13]:
val_results = [
    {'model': 'Naive-1',               **naive1_val},
    {'model': 'Weekly Seasonal Naive', **weekly_naive_val},
    {'model': 'MA-7',                  **ma7_val},
    {'model': 'MA-7 today-inclusive',  **ma7_incl_val},
    {'model': 'MA-28',                 **ma28_val},
]
val_df = (
    pd.DataFrame(val_results)
      .rename(columns={'wmape': 'WMAPE', 'mae': 'MAE',
                       'nv_cost': 'NV Cost/Row', 'n_rows': 'N'})
      # Primary: WMAPE ascending.  Secondary: MAE ascending (tie-break).
      # kind='stable' preserves insertion order for remaining ties.
      .sort_values(['WMAPE', 'MAE'], kind='stable')
      .reset_index(drop=True)
)
display(val_df[['model', 'WMAPE', 'MAE', 'NV Cost/Row', 'N']])

best_baseline = val_df.iloc[0]['model']
print(f'\n★  Best baseline (validation): {best_baseline}')
print(f"   WMAPE = {val_df.iloc[0]['WMAPE']:.4f}   MAE = {val_df.iloc[0]['MAE']:.4f}")
print('\n→  TEST will now be unlocked for this baseline only.')

# Explicit mapping: winner → (validation view, prediction column)
_val_view_map = {
    'Naive-1':               ('val',             'units_sold'),
    'Weekly Seasonal Naive': ('weekly_naive_val', 'weekly_naive_pred'),
    'MA-7':                  ('val',             'rolling_mean_7'),
    'MA-7 today-inclusive':  ('val_ma7incl',     'ma7_incl_pred'),
    'MA-28':                 ('val',             'rolling_mean_28'),
}
best_val_view, best_pred_col = _val_view_map[best_baseline]
print(f"Segment analysis will use: view='{best_val_view}', pred='{best_pred_col}'")

# Formally unlock the test set
TEST_UNLOCKED = True

,model,WMAPE,MAE,NV Cost/Row,N
0,MA-7 today-inclusive,0.483053,4.551775,3.656562,9849078
1,MA-28,0.491407,4.630493,3.820734,9849078
2,MA-7,0.494999,4.664343,3.780199,9849078
3,Naive-1,0.572872,5.398132,4.062765,9849078
4,Weekly Seasonal Naive,0.581036,5.475062,4.404866,9849078



★  Best baseline (validation): MA-7 today-inclusive
   WMAPE = 0.4831   MAE = 4.5518

→  TEST will now be unlocked for this baseline only.
Segment analysis will use: view='val_ma7incl', pred='ma7_incl_pred'


In [14]:
# Protocol gate — formal unlock event
assert TEST_UNLOCKED, (
    '[FAIL] TEST_UNLOCKED is False. '
    'Cell 12 must select a validation winner before test can be evaluated.'
)

# Create test view — bounded on both ends for an explicit contract
con.execute(f"""
    CREATE OR REPLACE VIEW test AS
    SELECT * FROM features
    WHERE target_date >= '{TEST_START}'
      AND target_date <= '{TEST_END}'
""")

# Verify test is non-empty and temporally after validation
test_rows = con.execute('SELECT COUNT(*) FROM test').fetchone()[0]
s_min     = con.execute('SELECT MIN(target_date) FROM test').fetchone()[0]
s_max     = con.execute('SELECT MAX(target_date) FROM test').fetchone()[0]
assert test_rows > 0, '[FAIL] Test split is empty'
assert v_max < s_min, f'[FAIL] Validation max {v_max} overlaps test min {s_min}'

# Full coverage check: train + val + test == total
split_total = train_rows + val_rows + test_rows
assert split_total == total_rows, (
    f'[FAIL] Split gap: {train_rows:,}+{val_rows:,}+{test_rows:,}'
    f'={split_total:,} ≠ total {total_rows:,}. Check for date gaps.'
)
print(f'[PASS] Coverage: {train_rows:,} + {val_rows:,} + {test_rows:,} = {total_rows:,}')
print(f'Test range: {s_min} → {s_max}')

# Create any derived test views required for the winning baseline
if best_baseline == 'Weekly Seasonal Naive':
    con.execute("""
        CREATE OR REPLACE VIEW weekly_naive_test AS
        SELECT
            a.forecast_date, a.store_id, a.sku_id, a.target_unit_sales,
            COALESCE(b.units_sold, 0.0) AS weekly_naive_pred,
            (b.units_sold IS NULL)      AS was_zero_filled
        FROM test a
        LEFT JOIN hist b
            ON  a.store_id      = b.store_id
            AND a.sku_id        = b.sku_id
            AND a.forecast_date = b.forecast_date + INTERVAL 6 DAYS
    """)
    test_view, test_pred = 'weekly_naive_test', 'weekly_naive_pred'

elif best_baseline == 'MA-7 today-inclusive':
    con.execute("""
        CREATE OR REPLACE VIEW test_ma7incl AS
        SELECT *,
            (rolling_mean_7 * 7 - lag_7 + units_sold) / 7.0 AS ma7_incl_pred
        FROM test
    """)
    test_view, test_pred = 'test_ma7incl', 'ma7_incl_pred'

else:
    _test_view_map = {
        'Naive-1': ('test', 'units_sold'),
        'MA-7':    ('test', 'rolling_mean_7'),
        'MA-28':   ('test', 'rolling_mean_28'),
    }
    test_view, test_pred = _test_view_map[best_baseline]

# ONE final test evaluation — winner only
best_test = compute_metrics_sql(test_view, test_pred)

print(f'\n{best_baseline}')
print(f"  VALIDATION  WMAPE={val_df.iloc[0]['WMAPE']:.4f}   MAE={val_df.iloc[0]['MAE']:.4f}")
print(f"  TEST        WMAPE={best_test['wmape']:.4f}   MAE={best_test['mae']:.4f}")

val_test_gap = abs(best_test['wmape'] - val_df.iloc[0]['WMAPE'])
if val_test_gap > 0.05:
    print(f'  ⚠ WARNING: val/test WMAPE gap = {val_test_gap:.4f}. '
          'Possible distribution shift in 2017 data.')

[PASS] Coverage: 73,711,181 + 9,849,078 + 19,571,966 = 103,132,225
Test range: 2017-01-01 → 2017-08-15

MA-7 today-inclusive
  VALIDATION  WMAPE=0.4831   MAE=4.5518
  TEST        WMAPE=0.4859   MAE=4.4298


In [15]:
# Demand tier thresholds computed from TRAIN ONLY — no test leakage
q33, q66 = con.execute("""
    SELECT
        QUANTILE_CONT(rolling_mean_28, 0.33),
        QUANTILE_CONT(rolling_mean_28, 0.66)
    FROM train
""").fetchone()

print(f'Demand tier thresholds (train rolling_mean_28 percentiles):')
print(f'  Low   :  rolling_mean_28 <= {q33:.3f}')
print(f'  Mid   :  {q33:.3f} < rolling_mean_28 <= {q66:.3f}')
print(f'  High  :  rolling_mean_28 > {q66:.3f}')
print('Note: thresholds frozen from train — applied identically to val/test.')

Demand tier thresholds (train rolling_mean_28 percentiles):
  Low   :  rolling_mean_28 <= 2.786
  Mid   :  2.786 < rolling_mean_28 <= 6.964
  High  :  rolling_mean_28 > 6.964
Note: thresholds frozen from train — applied identically to val/test.


In [16]:
# Segment analysis uses best_val_view / best_pred_col defined in Cell 12.
# All metric outputs CAST to DOUBLE — consistent with Cell 7 contract.
pred = best_pred_col
view = best_val_view

# 1. By store (54 rows)
store_df = con.execute(f"""
    SELECT
        store_id,
        CAST(SUM(ABS(target_unit_sales - {pred})) /
             NULLIF(SUM(ABS(target_unit_sales)), 0) AS DOUBLE) AS wmape,
        CAST(AVG(ABS(target_unit_sales - {pred})) AS DOUBLE)   AS mae,
        COUNT(*)                                                AS n
    FROM {view}
    WHERE {pred} IS NOT NULL
    GROUP BY store_id
    ORDER BY wmape DESC
""").df()
print('Worst 10 stores by WMAPE:')
display(store_df.head(10))

# 2. By product family
family_df = con.execute(f"""
    SELECT
        family,
        CAST(SUM(ABS(target_unit_sales - {pred})) /
             NULLIF(SUM(ABS(target_unit_sales)), 0) AS DOUBLE) AS wmape,
        CAST(AVG(ABS(target_unit_sales - {pred})) AS DOUBLE)   AS mae,
        COUNT(*)                                                AS n
    FROM {view}
    WHERE {pred} IS NOT NULL
    GROUP BY family
    ORDER BY wmape DESC
""").df()
display(family_df)

# 3-5. Combined segment table (perishability / promotion / demand tier)
segment_df = con.execute(f"""
    SELECT 'is_perishable' AS segment_type,
           CAST(is_perishable AS VARCHAR)                             AS segment_value,
           CAST(SUM(ABS(target_unit_sales - {pred})) /
               NULLIF(SUM(ABS(target_unit_sales)), 0) AS DOUBLE)     AS wmape,
           CAST(AVG(ABS(target_unit_sales - {pred})) AS DOUBLE)      AS mae,
           COUNT(*)                                                   AS n
    FROM {view} WHERE {pred} IS NOT NULL
    GROUP BY is_perishable

    UNION ALL

    SELECT 'promotion_tomorrow',
           CAST(on_promotion_tomorrow AS VARCHAR),
           CAST(SUM(ABS(target_unit_sales - {pred})) /
               NULLIF(SUM(ABS(target_unit_sales)), 0) AS DOUBLE),
           CAST(AVG(ABS(target_unit_sales - {pred})) AS DOUBLE),
           COUNT(*)
    FROM {view} WHERE {pred} IS NOT NULL
    GROUP BY on_promotion_tomorrow

    UNION ALL

    SELECT 'demand_tier',
           CASE WHEN rolling_mean_28 <= {q33} THEN 'low'
                WHEN rolling_mean_28 <= {q66} THEN 'mid'
                ELSE 'high' END,
           CAST(SUM(ABS(target_unit_sales - {pred})) /
               NULLIF(SUM(ABS(target_unit_sales)), 0) AS DOUBLE),
           CAST(AVG(ABS(target_unit_sales - {pred})) AS DOUBLE),
           COUNT(*)
    FROM {view} WHERE {pred} IS NOT NULL
    GROUP BY CASE WHEN rolling_mean_28 <= {q33} THEN 'low'
                  WHEN rolling_mean_28 <= {q66} THEN 'mid'
                  ELSE 'high' END
    ORDER BY segment_type, wmape DESC
""").df()
display(segment_df)

Worst 10 stores by WMAPE:


,store_id,wmape,mae,n
0,26,0.663858,3.456098,101615
1,35,0.628689,4.107599,88459
2,25,0.623661,4.441618,79499
3,32,0.616945,3.201280,83008
4,18,0.587917,3.989151,38865
5,22,0.575076,2.885727,125675
6,41,0.568836,4.033970,189454
7,13,0.559040,3.467827,122068
8,14,0.557842,3.693025,136314
9,16,0.557230,3.114926,119414


,family,wmape,mae,n
0,HOME AND KITCHEN I,0.736199,2.331238,27367
1,SCHOOL AND OFFICE SUPPLIES,0.694667,4.483131,3904
2,AUTOMOTIVE,0.688953,1.226326,9913
3,HARDWARE,0.680623,1.059235,1857
4,LINGERIE,0.679408,1.509768,8241
5,BABY CARE,0.659849,1.359202,451
6,CELEBRATION,0.652772,1.613553,24580
7,LADIESWEAR,0.650508,1.666451,17023
8,FROZEN FOODS,0.632338,9.566459,105910
9,"LIQUOR,WINE,BEER",0.626199,5.245117,72929


,segment_type,segment_value,wmape,mae,n
0,demand_tier,low,0.610064,1.831035,3645530
1,demand_tier,mid,0.537514,3.081170,3151301
2,demand_tier,high,0.445852,9.319691,3052247
3,is_perishable,false,0.513852,4.421468,7149498
4,is_perishable,true,0.422497,4.896876,2699580
5,promotion_tomorrow,true,0.483184,6.799815,1398332
6,promotion_tomorrow,false,0.483017,4.179795,8450746


In [17]:
# Note: pyarrow.parquet not imported — DataFrame.to_parquet() uses default pyarrow engine.

# 1. Overall metrics — all 5 baselines (val) + winner (test)
overall_rows = [
    {'model': r['model'], 'split': 'validation',
     'wmape': r['WMAPE'], 'mae': r['MAE'],
     'nv_cost': r['NV Cost/Row'], 'n_rows': int(r['N'])}
    for _, r in val_df.iterrows()
] + [
    {'model': best_baseline, 'split': 'test',
     'wmape': best_test['wmape'], 'mae': best_test['mae'],
     'nv_cost': best_test['nv_cost'], 'n_rows': best_test['n_rows']}
]
pd.DataFrame(overall_rows).to_parquet(
    OUTPUT_DIR / 'overall_metrics.parquet', index=False)

# 2. Store metrics
store_df.to_parquet(OUTPUT_DIR / 'store_metrics.parquet',   index=False)

# 3. Family metrics
family_df.to_parquet(OUTPUT_DIR / 'family_metrics.parquet', index=False)

# 4. Combined segment metrics
segment_df.to_parquet(OUTPUT_DIR / 'segment_metrics.parquet', index=False)

# 5. Deterministic prediction sample — 1M rows, ORDER BY for reproducibility
#    This is a DIAGNOSTIC SAMPLE — metrics were computed over the full val set.
pred_sample = con.execute(f"""
    SELECT
        v.forecast_date, v.store_id, v.sku_id, v.target_unit_sales,
        v.units_sold                                          AS naive_pred,
        COALESCE(wn.weekly_naive_pred, 0.0)                   AS weekly_naive_pred,
        v.rolling_mean_7                                      AS ma7_pred,
        (v.rolling_mean_7*7 - v.lag_7 + v.units_sold) / 7.0  AS ma7_incl_pred,
        v.rolling_mean_28                                     AS ma28_pred
    FROM val v
    LEFT JOIN weekly_naive_val wn
        USING (forecast_date, store_id, sku_id)
    ORDER BY v.forecast_date, v.store_id, v.sku_id
    LIMIT 1000000
""").df()
pred_sample.to_parquet(OUTPUT_DIR / 'baseline_predictions.parquet', index=False)
print(f'Saved deterministic sample: {len(pred_sample):,} rows '
      f'(diagnostic only — metrics computed over full val set in DuckDB)')

# 6. Baseline contract JSON — consumed by Notebook 04
# Distinguishes: (a) validation model-selection threshold
#            vs (b) final test comparison target
baseline_contract = {
    'schema_version': 1,
    'produced_by':    '03_baselines_and_evaluation',
    'run_timestamp':  datetime.now(timezone.utc).isoformat(),
    'split': {
        'key':              'target_date',
        'train_end':        TRAIN_END,
        'validation_start': VAL_START,
        'validation_end':   VAL_END,
        'test_start':       TEST_START,
        'test_end':         TEST_END,
    },
    'newsvendor': {
        'cu':   CU,
        'co':   CO,
        'note': 'Temporary analytical assumptions — not validated business costs',
    },
    'primary_metric': 'WMAPE',
    'best_baseline':  best_baseline,
    # Model selection: NB04 must beat this on VALIDATION
    'model_selection': {
        'split':           'validation',
        'threshold_wmape': float(val_df.iloc[0]['WMAPE']),
        'threshold_mae':   float(val_df.iloc[0]['MAE']),
        'rule': 'LightGBM validation WMAPE must be strictly less than threshold_wmape.',
    },
    # Final evaluation: NB04 compares against this on TEST
    'final_evaluation': {
        'split':          'test',
        'baseline_wmape': float(best_test['wmape']),
        'baseline_mae':   float(best_test['mae']),
        'rule': (
            'Final LightGBM test WMAPE compared against baseline_wmape. '
            'The model is evaluated on test ONCE, after all hyperparameter '
            'decisions are finalized using validation only.'
        ),
    },
    'protocol': {
        'training_data':        f'target_date <= {TRAIN_END}',
        'model_selection_data': f'{VAL_START} <= target_date <= {VAL_END}',
        'holdout_data':         f'{TEST_START} <= target_date <= {TEST_END}',
        'leakage_rule': (
            'target_unit_sales must not be used as input feature. '
            'All features are scoped to forecast_date D. Target is D+1.'
        ),
    },
}
with open(OUTPUT_DIR / 'baseline_contract.json', 'w') as f:
    json.dump(baseline_contract, f, indent=2, default=str)

print('\n[OK] All outputs saved:')
for p in sorted(OUTPUT_DIR.iterdir()):
    kb = p.stat().st_size / 1024
    print(f'     {p.name:<40} {kb:>8.1f} KB')

Saved deterministic sample: 1,000,000 rows (diagnostic only — metrics computed over full val set in DuckDB)

[OK] All outputs saved:
     baseline_contract.json                        1.4 KB
     baseline_predictions.parquet              12923.4 KB
     family_metrics.parquet                        3.9 KB
     overall_metrics.parquet                       4.0 KB
     segment_metrics.parquet                       3.5 KB
     store_metrics.parquet                         4.1 KB


## Baseline Evaluation Complete

| Split | target_date range |
|---|---|
| Train | ≤ 2016-08-31 |
| Validation | 2016-09-01 → 2016-12-31 |
| Test | 2017-01-01 → max |

### Results

See `forecasting_baseline_results/overall_metrics.parquet` for full metrics.

### Newsvendor Cost

Cu=1.0, Co=0.5 are **temporary analytical assumptions** — not validated business costs.  
Replace with actual margin and waste data when available.

### Notebook 04 Requirement

See `forecasting_baseline_results/baseline_contract.json` for the typed contract.

```
LightGBM must achieve validation WMAPE < best baseline validation WMAPE
Trained on  : target_date <= 2016-08-31
Selected on : 2016-09-01 <= target_date <= 2016-12-31  (validation only)
Evaluated on: 2017-01-01 <= target_date <= max         (once, after selection)
Zero leakage: target_unit_sales must not appear in features
```

**WMAPE note:** values > 100% are valid — WMAPE is not capped at 1.0.